#Database & Understanding| Lina Ghawanmeh

In [ ]:
# Import PySpark libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

# Create Spark Session
spark = SparkSession.builder \
    .appName("OnlineRetailAnalysis") \
    .getOrCreate()

    # Read CSV file
df = spark.read.csv(
    "/content/online_retail.csv",
    header=True,
    inferSchema=True
)

# Display first 5 rows
df.show(5)

# Print schema
df.printSchema()

# Number of rows and columns
print("Rows:", df.count())
print("Columns:", len(df.columns))

# Data Cleaning| Rawaa Alhorani

In [ ]:
# Check missing values
from pyspark.sql.functions import col, sum
df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).show()

In [ ]:
# Check duplicate rows
duplicates = df.count() - df.dropDuplicates().count()

print("Duplicate Rows:", duplicates)

In [ ]:
# Remove rows with missing values
df = df.dropna()

In [ ]:
# Remove duplicate rows
df = df.dropDuplicates()

In [ ]:
# Display cleaned data
df.show(5)

# Data Preparation| Layali Alazzam

In [ ]:
# Create total price column
df = df.withColumn(
    "TotalPrice",
    col("Quantity") * col("UnitPrice")
)

df.show(5)

In [ ]:
# Total sales by country
df.groupBy("Country") \
    .sum("TotalPrice") \
    .orderBy(col("sum(TotalPrice)").desc()) \
    .show()

In [ ]:
# Most sold products
df.groupBy("Description") \
    .sum("Quantity") \
    .orderBy(col("sum(Quantity)").desc()) \
    .show()

# Processing (PySpark Operations)| Dana Nawasreh

In [ ]:
# Filter products with quantity greater than 10
df.filter(col("Quantity") > 10)

# Filter orders from United Kingdom
df.filter(col("Country") == "United Kingdom")

In [ ]:
# Count orders by country
df.groupBy("Country") \
    .count() \
    .show()

In [ ]:
# Average unit price by country
df.groupBy("Country") \
    .avg("UnitPrice") \
    .show()

In [ ]:
# Sort products by unit price
df.orderBy(col("UnitPrice").desc()).show()

In [ ]:
# Save cleaned data
df.toPandas().to_csv(
    "cleaned_online_retail.csv",
    index=False
)

# Analysis & Results| Batool Alghazawi

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 130
plt.rcParams['font.size'] = 11
COLORS = {'churn': '#e74c3c', 'no_churn': '#2ecc71', 'accent': '#3498db'}

In [ ]:
from pyspark.sql.functions import when, col, sum
churn_counts_df_spark = df.groupBy('Churn').count()
churn_counts_pd = churn_counts_df_spark.toPandas()
churn_counts_pd = churn_counts_pd.sort_values(by='Churn', ascending=True)
churn_counts = churn_counts_pd['count'].values
fig, ax = plt.subplots(figsize=(7, 7))
wedges, texts, autotexts = ax.pie(
    churn_counts,
    labels=['Stayed', 'Churned'],
    autopct='%1.1f%%',
    colors=[COLORS['no_churn'], COLORS['churn']],
    startangle=90,
    wedgeprops=dict(width=0.5)
)
for at in autotexts:
    at.set_fontweight('bold'); at.set_fontsize(13)
ax.set_title('Customer Churn Distribution', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('graph1_churn_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
total = df.count()
df_with_churn_label = df.withColumn('Churn_Label', when(col('Churn') == 'Yes', 1).otherwise(0))
churned = df_with_churn_label.agg(sum('Churn_Label')).collect()[0][0]
print(f"Finding: {churned:,} out of {total:,} customers churned "
      f"({churned/total*100:.1f}% churn rate).")

In [ ]:
df_pd = df_with_churn_label.toPandas()
contract_churn= df_pd.groupby('Contract')['Churn_Label'].mean() * 100
contract_churn = contract_churn.sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(contract_churn.index, contract_churn.values,
              color=[COLORS['churn'], COLORS['accent'], COLORS['no_churn']],
              edgecolor='white', linewidth=1.5, width=0.5)
for bar, val in zip(bars, contract_churn.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', fontweight='bold', fontsize=11)
ax.set_title('Churn Rate by Contract Type', fontsize=14, fontweight='bold')
ax.set_ylabel('Churn Rate (%)')
ax.set_ylim(0, contract_churn.max() * 1.25)
ax.grid(axis='y', alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('graph2_churn_by_contract.png', dpi=150, bbox_inches='tight')
plt.show()
best = contract_churn.idxmin()
worst = contract_churn.idxmax()
print(f"Finding: '{worst}' contract has the highest churn rate "
      f"({contract_churn[worst]:.1f}%), while '{best}' contract "
      f"has the lowest ({contract_churn[best]:.1f}%).")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(df_pd[df_pd['Churn']=='No']['MonthlyCharges'],
        bins=35, alpha=0.6, color=COLORS['churn'], label='Stayed', edgecolor='white')
ax.hist(df_pd[df_pd['Churn']=='Yes']['MonthlyCharges'],
        bins=35, alpha=0.6, color=COLORS['churn'],    label='Churned', edgecolor='white')
avg_stayed  = df_pd[df_pd['Churn']=='No']['MonthlyCharges'].mean()
avg_churned = df_pd[df_pd['Churn']=='Yes']['MonthlyCharges'].mean()
ax.axvline(avg_stayed,  color=COLORS['no_churn'], linestyle='--', linewidth=2,
           label=f'Stayed Avg: ${avg_stayed:.0f}')
ax.axvline(avg_churned, color=COLORS['churn'],    linestyle='--', linewidth=2,
           label=f'Churned Avg: ${avg_churned:.0f}')

ax.set_title('Monthly Charges: Churned vs Stayed', fontsize=14, fontweight='bold')
ax.set_xlabel('Monthly Charges ($)'); ax.set_ylabel('Number of Customers')
ax.legend(); ax.grid(axis='y', alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('graph3_monthly_charges.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
df_pd['tenure_group'] = pd.cut(df_pd['tenure'],
                                bins=[0, 12, 24, 48, df_pd['tenure'].max()],
                                labels=['0-12 months', '13-24 months', '25-48 months', '49+ months'],
                                right=False)
tenure_order = ['0-12 months', '13-24 months', '25-48 months', '49+ months']
tenure_churn = df_pd.groupby('tenure_group')['Churn_Label'].mean() * 100
tenure_churn = tenure_churn.reindex(tenure_order)
fig, ax = plt.subplots(figsize=(8, 5))
colors_t = [COLORS['churn'], '#e67e22', COLORS['accent'], COLORS['no_churn']]
bars = ax.bar(range(len(tenure_order)), tenure_churn.values,
              color=colors_t, edgecolor='white', linewidth=1.5, width=0.55)
for bar, val in zip(bars, tenure_churn.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}%', ha='center', fontweight='bold', fontsize=11)
    ax.set_xticks(range(len(tenure_order)))
    ax.set_xticklabels(tenure_order, fontsize=10)

    ax.set_title('Churn Rate by Customer Tenure', fontsize=14, fontweight='bold')
    ax.set_ylabel('Churn Rate (%)')
    ax.set_ylim(0, tenure_churn.max() * 1.25)
    ax.grid(axis='y', alpha=0.3)
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.savefig('graph4_churn_by_tenure.png', dpi=150, bbox_inches='tight')
    plt.show()

    print( f"Finding: New customers (0-12 months) have the highest churn rate "
      f"({tenure_churn['0-12 months']:.1f}%), dropping significantly "
      f"to {tenure_churn['49+ months']:.1f}% for long-term customers (49+ months).")





In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import pyspark.sql.functions as F
from pyspark.sql.functions import col
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 130
plt.rcParams['font.size'] = 11
COLORS = {'churn': '#e74c3c', 'no_churn': '#2ecc71', 'accent': '#3498db'}

In [ ]:
churn_data = df.groupBy("Churn").count().collect()

churn_dict = {
    row["Churn"]: row["count"]
        for row in churn_data
        }

        labels = ["Stayed", "Churned"]

        values = [
            churn_dict.get("No", 0),
                churn_dict.get("Yes", 0)
                ]

                fig, ax = plt.subplots(figsize=(7,7))

                ax.pie(
                    values,
                        labels=labels,
                            autopct='%1.1f%%',
                                startangle=90,
                                    colors=[
                                            COLORS['no_churn'],
                                                    COLORS['churn']
                                                        ]
                                                        )

                                                        ax.set_title("Customer Churn Distribution")

                                                        plt.show()

In [ ]:
total = sum(values)
churned = churn_dict.get("Yes", 0)

print("Analysis:")
print(f"{churned} customers churned out of {total}")
print(f"Churn Rate = {round(churned/total*100,1)}%")

In [ ]:
contract_data = df.groupBy("Contract") \
    .agg(
            F.round(
                        F.avg("Churn_Label") * 100,
                                    1
                                            ).alias("churn_rate")
                                                ) \
                                                    .orderBy(F.col("churn_rate").desc()) \
                                                        .collect()

                                                        contracts = [
                                                            row["Contract"]
                                                                for row in contract_data
                                                                ]

                                                                rates = [
                                                                    row["churn_rate"]
                                                                        for row in contract_data
                                                                        ]

                                                                        fig, ax = plt.subplots(figsize=(8,5))

                                                                        bars = ax.bar(
                                                                            contracts,
                                                                                rates,
                                                                                    color=[
                                                                                            COLORS['churn'],
                                                                                                    COLORS['accent'],
                                                                                                            COLORS['no_churn']
                                                                                                                ]
                                                                                                                )

                                                                                                                ax.set_title("Churn Rate by Contract Type")
                                                                                                                ax.set_ylabel("Churn Rate                                                                  plt.show()

In [ ]:
import pyspark.sql.functions as F
df_with_churn_label = df.withColumn('Churn_Label', when(col('Churn') == 'Yes', 1).otherwise(0))
contract_data = df_with_churn_label.groupBy("Contract") \
    .agg(
            F.round(
                        F.avg("Churn_Label") * 100,
                                    1
                                            ).alias("churn_rate")
                                                ) \
                                                    .orderBy(F.col("churn_rate").desc()) \
                                                        .collect()

contracts = [
    row["Contract"]
        for row in contract_data
        ]

rates = [
    row["churn_rate"]
        for row in contract_data
        ]

fig, ax = plt.subplots(figsize=(8,5))

bars = ax.bar(
    contracts,
        rates,
            color=[
                    COLORS['churn'],
                            COLORS['accent'],
                                    COLORS['no_churn']
                                        ]
                                        )

ax.set_title("Churn Rate by Contract Type")
ax.set_ylabel("Churn Rate (%)")

plt.show()

In [ ]:
print("📌 Analysis:")
print(f"Highest churn rate: {contracts[0]} ({rates[0]}%)")
print(f"Lowest churn rate: {contracts[-1]} ({rates[-1]}%)")